# 14장 · LCEL 기초 · 프롬프트, 모델, 출력 파서 연결하기

13장에서 공지 한 건과 질문 한 건을 저장했습니다. 이번에는 두 문자열이 실제로 모델에 보낼 메시지가 되는 줄을 확인합니다.

LangChain에서는 프롬프트로 메시지를 만들고, 모델에 보내 답을 받은 뒤, 출력 파서로 답의 글을 꺼낼 수 있습니다. 먼저 모델 없이 메시지 하나를 만들어 봅니다.

## ChatPromptTemplate.invoke로 모델에 보낼 메시지 만들기

`ChatPromptTemplate`은 문장의 일부를 빈칸으로 둔 채팅 메시지 양식입니다. `from_messages()`에 메시지의 역할과 문장을 적고, `invoke()`에 빈칸을 채울 값을 넘깁니다. 다음 코드에서 `human`은 사용자가 보낼 말이고 `{word}`가 빈칸입니다.

이 셀에는 API 키가 필요 없습니다. LangChain을 처음 설치하는 커널이라면 먼저 별도 셀에서 `%pip install langchain-core`를 실행하세요.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([("human", "{word}를 영어로 번역해 줘.")])
ready = prompt.invoke({"word": "사과"})
print(ready.to_messages()[0].content)

사과를 영어로 번역해 줘.


화면에는 “사과를 영어로 번역해 줘.”가 나옵니다. `ready`에는 모델에 보낼 메시지가 들어 있습니다. `to_messages()`는 메시지 목록을 돌려주고, `[0].content`는 첫 메시지의 본문을 꺼냅니다. 번역한 답은 뒤에서 모델을 호출할 때 받습니다.

### 공지에서 만든 질문을 프롬프트에 넣어 보기

1~42장 실습 묶음을 풀고 같은 폴더에서 실행하세요. 내 공지와 질문은 `my-course/chat-case.json`에 있습니다. 파일이 없으면 제공한 305호 공지로 실습합니다.

In [ ]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pre-mid/documents/rag/rag_lab.py").is_file()), None)
if root is None:
    raise FileNotFoundError("1~42장 실습 묶음을 모두 풀고 그 폴더에서 시작하세요.")
rag_dir = root / "pre-mid/documents/rag"
sys.path.insert(0, str(rag_dir))
from portfolio import chat_case, save_step
case = chat_case(root)
template = "질문: {question}\n자료: {context}"
notice_prompt = ChatPromptTemplate.from_messages([("human", template)])
packet = notice_prompt.invoke(case).to_messages()[0]
print(packet.content)

13장 완성 예시 사용: chat-case.json
질문: 어디에서 모이나요?
자료: 5월 21일 오후 4시에 가온관 305호에서 모입니다.


`invoke(case)`는 내 질문과 공지를 각각 같은 이름의 빈칸에 넣습니다. 방금 출력한 두 줄을 읽어 보세요. 질문을 짧게 써도 바로 다음 줄에서 장소를 찾을 수 있습니다. 15장에서는 두 번째 줄에 PDF 원문을 넣습니다.

In [ ]:
save_step(root, "prompt", {"template": template, "case": case})
print("my-course/rag/prompt.json")

my-course/rag/prompt.json


### 실습 · 입력이 빠진 곳 찾기

공지 문장만 빈 문자열로 바꾼 메시지를 만드세요. 질문은 그대로이고 마지막 줄에는 장소가 없어야 합니다. 저장 파일을 덮어쓰지 않고 새 입력을 만들어 비교하세요.

In [ ]:
changed = {**case, "context": ""}
# 같은 양식에 changed를 넣고 마지막 줄의 내용을 검사하세요.

<details><summary>정답 코드·출력·풀이</summary><pre><code>text = notice_prompt.invoke(changed).to_messages()[0].content
print(case[&quot;question&quot;] in text, text.endswith(&quot;자료: &quot;))</code></pre><p>제공 자료로 확인할 값</p><pre>True True</pre><p>빈 문자열도 유효한 입력입니다. 질문이 남았다는 사실과 답할 근거가 남았는지는 따로 확인해야 합니다.</p></details>

### 같은 양식에 다른 값 넣기

이번에는 양식을 그대로 두고 `word`의 값만 바꿉니다.

In [ ]:
next_message = prompt.invoke({"word": "포도"})
print(next_message.to_messages()[0].content)

포도를 영어로 번역해 줘.


“포도”만 바뀌었습니다. `from_messages()`는 양식을 만들 때 한 번, `invoke()`는 새 값을 채울 때마다 사용합니다.

## KeyError가 나면 빈칸 이름 확인하기

`{"fruit": "사과"}`를 넘기면 `{word}`를 채울 값이 없습니다. 어떤 이름이 필요한지는 `prompt.input_variables`에서 확인할 수 있습니다.

In [ ]:
try:
    prompt.invoke({"fruit": "사과"})
except KeyError:
    print("필요한 키:", prompt.input_variables)

필요한 키: ['word']


출력에 나온 이름대로 딕셔너리 키를 `word`로 고치면 됩니다. 오류는 메시지를 만드는 단계에서 났으므로 모델 설정을 바꿀 필요가 없습니다.

## 실습 · 빈칸 두 개를 가진 메시지 만들기

번역할 단어와 언어를 모두 바꿀 수 있게 `{word}`와 `{language}`를 둔 양식을 만드세요. “빵”과 “프랑스어”를 넣고, 모델에 보내기 직전의 문장을 출력하세요.

In [ ]:
values = {"word": "빵", "language": "프랑스어"}
# 두 키를 빈칸으로 쓰는 human 메시지 양식을 만들고 invoke(values)로 채우세요.
# to_messages()[0].content로 모델에 전달할 한 문장을 출력하세요.

<details class="notebook-answer"><summary>정답 코드·실행 결과·풀이</summary><pre><code>translation = ChatPromptTemplate.from_messages([(&quot;human&quot;, &quot;{word}을 {language}로 번역해 줘.&quot;)])
prepared = translation.invoke({&quot;word&quot;: &quot;빵&quot;, &quot;language&quot;: &quot;프랑스어&quot;})
print(prepared.to_messages()[0].content)</code></pre><pre>빵을 프랑스어로 번역해 줘.</pre><p>두 키가 각각 같은 이름의 빈칸을 채웠습니다. 출력한 것은 모델에 보낼 문장이므로 실제 번역을 하려면 이 메시지를 모델에 넘겨야 합니다.</p></details>

### StrOutputParser에 짧은 응답 메시지 넣기

`StrOutputParser().invoke(메시지)`는 응답 메시지에서 글을 꺼냅니다. 아래에서는 직접 만든 메시지를 넣어 반환 자료형을 확인합니다. 이 셀은 모델을 부르지 않습니다.

In [ ]:
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import StrOutputParser

message = AIMessage(content="apple")
word = StrOutputParser().invoke(message)
print(word, isinstance(word, str))

apple True


첫 값 `apple`은 본문입니다. `True`는 그 결과를 파이썬에서 문자열로 사용할 수 있다는 뜻입니다. 다음에는 모델이 실제로 반환한 메시지를 같은 파서에 넣습니다.

## model.invoke로 메시지 보내기

`model.invoke(ready)`는 앞에서 만든 메시지를 모델에 보내고 답을 메시지 객체로 받습니다. 이제부터는 실제 API 키가 필요합니다. 8장의 환경을 쓰거나 별도 셀에서 `%pip install langchain-openai python-dotenv`를 실행한 뒤 작업 폴더의 `.env`에 `OPENAI_API_KEY`를 설정하세요.

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
model = ChatOpenAI(model="gpt-4.1-mini")
reply = model.invoke(ready)
print(reply.content)

여기서 처음으로 번역한 답이 나옵니다. “apple”을 포함하는지 보세요. 모델은 설명을 덧붙일 수도 있습니다. `ready`는 보낸 메시지이고, `reply`는 받은 답입니다.

## 실제 응답을 StrOutputParser에 넘기기

`StrOutputParser`는 응답 메시지에서 글을 꺼냅니다. 앞 셀에서 받은 `reply`를 그대로 넘기므로 번역을 다시 요청하지 않습니다.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
answer = parser.invoke(reply)
print(answer)

화면에 보이는 글은 `reply.content`를 출력했을 때와 같습니다. 이제 `answer`를 다른 문자열과 이어 붙일 수 있습니다. 사용량은 파싱하기 전의 `reply`에서 읽습니다.

### 한 번의 요청에서 사용량 읽기

`reply.usage_metadata`에는 모델이 보고한 토큰 수가 들어갑니다. 다음 셀은 이미 받은 응답을 읽습니다.

In [ ]:
print(reply.usage_metadata)

`input_tokens`는 보낸 메시지, `output_tokens`는 생성한 답에 쓰인 양입니다. 모델과 실제 답에 따라 값이 달라집니다.

## 파이프 연산자로 조립하고 chain.invoke로 실행하기

LCEL의 `|` 연산자는 왼쪽 단계의 출력을 오른쪽 단계의 입력에 연결합니다. `prompt | model | parser`로 연결하면 딕셔너리 하나를 넣어서 번역한 글을 받을 수 있습니다.

In [ ]:
chain = prompt | model | parser
print(chain.invoke({"word": "포도"}))

첫 줄은 실행 순서를 정합니다. 둘째 줄에서 “포도를 영어로 번역해 줘.”라는 메시지가 만들어지고, 모델이 번역한 뒤 파서가 본문을 꺼냅니다. 앞의 “사과” 요청과는 별도 호출입니다.

## temperature와 max_tokens로 생성 조건 바꾸기

`temperature`는 다음 토큰을 고를 때 후보 간 확률 차이에 영향을 줍니다. 낮추면 가능성이 높은 표현에 더 모이고 높이면 다른 표현도 나올 여지가 커집니다. 0에서도 응답이 매번 같다고 보장할 수 없습니다. `max_tokens`는 생성할 수 있는 토큰 수의 상한입니다. 아래에서는 연필 사진에 붙일 제목을 요청하면서 한 번에 한 설정만 바꿉니다. 예제 모델은 `gpt-4.1-mini`입니다.

In [ ]:
from langchain_openai import ChatOpenAI

title_request = '탁자 위에 노란 연필 한 자루가 놓여 있습니다. 이 장면의 제목을 한국어로 여섯 가지 써 주세요. 한 줄에 제목 하나만 쓰세요.'
# 160은 생성할 수 있는 최대 토큰 수입니다. 실제로 쓴 양은 응답에서 확인합니다.
editor = ChatOpenAI(model="gpt-4.1-mini", temperature=0, max_tokens=160)
plain = editor.invoke(title_request)
print(plain.content)

노란 연필의 고요한 순간  
탁자 위의 노란 연필  
고요한 책상과 노란 연필  
단 하나의 노란 연필  
탁자 위의 작은 노란 빛  
노란 연필과 정적의 공간


여섯 줄에서 ‘노란 연필’이 얼마나 되풀이되는지 먼저 읽으세요. 아래의 `bind`는 이번 호출에 적용할 설정을 붙입니다. 기존 `editor`와 질문을 유지하고 온도 하나만 바꿀 수 있습니다.

In [ ]:
# 같은 모델, 같은 요청, 같은 출력 상한에서 온도만 바꿉니다.
varied = editor.bind(temperature=0.9).invoke(title_request)
print(varied.content)

노란 연필과 고요한 탁자  
탁자 위의 노란 연필  
조용한 공간의 노란 연필  
하나뿐인 노란 연필  
탁자 위 노란 연필의 이야기  
노란 연필, 그 자리에서


이번 실제 실행에서는 두 답에 모두 ‘탁자 위의 노란 연필’이 있었습니다. 높은 온도에서 생긴 ‘하나뿐인 노란 연필’도 원문의 ‘한 자루’와 대조할 수 있습니다. 두 답이 얼마나 다채롭게 느껴지는지와 원문의 사실을 지켰는지는 따로 보세요. 한 번의 비교에서 특정 표현이 나온다는 규칙을 만들 수는 없습니다.

### 여섯 줄을 부탁했는데 두 번째 줄에서 끊겼다면?

`max_tokens`는 학생이 사용하는 `ChatOpenAI`의 **출력 상한**입니다. 8장의 OpenAI Responses API에서 대응하는 이름은 `max_output_tokens`입니다. 모델이 최대 몇 토큰까지 생성할지 지정하며, 답변이 짧게 끝나면 남은 분량을 채우지 않습니다. 이번에는 온도를 다시 0에 두고 상한만 줄입니다.

In [ ]:
short_editor = ChatOpenAI(model="gpt-4.1-mini", temperature=0, max_tokens=12)
short = short_editor.invoke(title_request)
print(short.content)
# Chat Completions 응답에서 생성이 끝난 이유와 실제 출력 사용량을 함께 읽습니다.
print(short.response_metadata["finish_reason"], short.usage_metadata["output_tokens"])

노란 연필의 고요한 순간  
탁자
length 12


첫 제목 뒤에 ‘탁자’만 남았습니다. 이 실행에서 `length`와 출력 12가 함께 나온 것은 설정한 상한에 도달했다는 단서입니다. 제목을 여섯 개 받아야 한다면 상한을 넉넉히 두고 다시 확인하세요. 반대로 한 개만 필요하면 요청 문장에서 개수를 고칩니다. 응답 시간을 제한하는 `timeout`은 초 단위이며, `max_retries`는 연결 오류 등 재시도 가능한 실패를 SDK가 몇 번 다시 시도할지 정합니다. 두 설정 모두 잘린 답을 이어 쓰는 기능은 아닙니다.

### 직접 해 보기 · 어디를 고칠까요?

① 제목 한 줄만 필요했는데 여섯 줄이 모두 왔습니다. ② 안내문에 신청 마감일이 들어가야 하는데 문장 중간에서 끊겼습니다. 두 경우에 처음 고칠 곳을 적고, 두 번째 경우에는 어떤 응답 필드를 열어볼지 적으세요. 그런 다음 첫 셀의 질문을 ‘제목 한 개’로 고치고 상한 160에서 실제 출력 토큰을 읽어 보세요.

<details class="notebook-answer"><summary>모범 판단 펼치기</summary><p>① 요청에 ‘한국어 제목 하나만’이라고 씁니다. ② 먼저 finish_reason과 output_tokens를 봅니다. length이고 상한에 닿았다면 상한을 올립니다. stop으로 자연스럽게 끝났는데 마감일만 빠졌다면 원문과 지시문에 마감일 보존 조건이 있는지 살핍니다. 상한 160은 한 줄 제목이 반드시 160토큰이어야 한다는 뜻이 아닙니다.</p></details>

## InMemoryCache로 같은 요청의 답 재사용하기

`InMemoryCache()`는 모델 호출 결과를 현재 프로세스의 메모리에 보관합니다. `ChatOpenAI(cache=...)`에 넘긴 뒤 같은 설정과 입력으로 두 번 호출해 보세요. 빈 캐시에서 시작하면 첫 번째 호출이 답을 저장하고 두 번째 호출이 그 답을 읽습니다.

In [ ]:
from langchain_core.caches import InMemoryCache

saved = ChatOpenAI(model="gpt-4.1-mini", temperature=0, max_tokens=160, cache=InMemoryCache())
first_title = saved.invoke(title_request)  # 빈 캐시이므로 모델에 보냅니다.
again = saved.invoke(title_request)        # 같은 요청의 저장된 응답을 읽습니다.
print(first_title.content == again.content)

True


집필 시 나가는 모델 요청을 세어 보니 첫 번째는 1건, 두 번째는 0건이었습니다. 같은 문장인지 비교한 `True`만으로는 호출 여부를 증명할 수 없습니다. 캐시에서 받은 메시지에 보이는 토큰 수는 첫 응답의 기록일 수 있습니다. 이를 새 호출의 사용량으로 더하면 두 번 계산하게 됩니다.

**실습**. 위 셀을 다시 실행하지 말고 새 셀에서 `saved.invoke(title_request)`를 실행해 보세요. 이어 요청 끝에 ‘각 제목에 번호를 붙이세요’를 추가해 보세요. 둘 중 어느 요청이 새 답을 받아야 할까요? 커널을 재시작했다면 앞서 보관한 값은 어떻게 될까요?

<details class="notebook-answer"><summary>모범 판단 펼치기</summary><p>같은 모델과 같은 입력은 저장된 답을 읽습니다. 번호를 요청하면 입력이 달라져 처음 한 번은 새로 호출합니다. 커널을 재시작하거나 InMemoryCache()를 새로 만들면 빈 메모리에서 시작합니다. 원문이 바뀌는 수업에서는 원문 자체도 모델 입력에 포함해야 새 판본을 다른 요청으로 구별할 수 있습니다.</p></details>

[ChatOpenAI 설정과 사용량](https://reference.langchain.com/python/langchain-openai/chat_models/base/ChatOpenAI) · [LangChain 메모리 캐시](https://reference.langchain.com/python/langchain-core/caches/InMemoryCache)